# Phase 0.3 — Training Stack Test

This notebook tests the complete training stack for Project Second.
It runs on Google Colab with an **NVIDIA L4 GPU**.

### Objectives
1. Verify GPU availability and memory (NVIDIA L4, 24 GB VRAM).
2. Mount Google Drive for persistent checkpoints.
3. Install dependencies (`unsloth`, `trl`, `transformers`, `accelerate`, `peft`, `datasets`).
4. Install fast attention kernels (`flash-linear-attention`, `causal-conv1d`) with fallback.
5. Load `Qwen/Qwen3.5-2B` in text-only non-thinking mode with LoRA.
6. Run 10 steps of Supervised Fine-Tuning (SFT) on dummy chess data in the Second format.
7. Run 10 steps of Group Relative Policy Optimization (GRPO) with dummy rule reward.
8. Test vLLM fast generation and verify standard TRL fallback.
9. Save a checkpoint to Google Drive.
10. Verify resumption from the saved checkpoint.
11. Output benchmark metrics for `docs/decisions/compute.md`.


In [ ]:
# Step 1: Check GPU hardware
!nvidia-smi

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError("No GPU detected. In Colab menu: Runtime -> Change runtime type -> L4 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU Name: {device_name}")
print(f"Total VRAM: {vram_gb:.2f} GB")

if "L4" not in device_name:
    print("Note: Phase 0.3 targets an NVIDIA L4 GPU. Other GPUs may have different timing.")


In [ ]:
# Step 2: Mount Google Drive for persistent checkpoint storage
import os
from pathlib import Path

CHECKPOINT_DIR = "/content/drive/MyDrive/second/checkpoints/phase0_test"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Google Drive mounted successfully.")
    print(f"Checkpoint directory: {CHECKPOINT_DIR}")
except Exception as exc:
    print(f"Google Drive mount notice: {exc}")
    CHECKPOINT_DIR = "/content/second_checkpoints/phase0_test"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Using local fallback directory: {CHECKPOINT_DIR}")


In [ ]:
# Step 3: Install dependencies and kernels with fallback handling
!pip install -q --upgrade pip
!pip install -q "transformers>=4.49.0" "trl>=0.15.0" "accelerate>=1.2.0" "peft>=0.14.0" "datasets>=3.0.0" "bitsandbytes>=0.45.0"

# Attempt Unsloth installation
try:
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    print("Unsloth installed successfully.")
except Exception as exc:
    print(f"Unsloth installation skipped: {exc}")

# Optional fast kernels (use prebuilt wheels only to avoid long C++ compilation)
print("Checking for prebuilt fast kernels...")
!pip install -q causal-conv1d --only-binary :all: 2>/dev/null || echo "causal-conv1d prebuilt binary not found. PyTorch fallback active."
!pip install -q flash-linear-attention --only-binary :all: 2>/dev/null || echo "flash-linear-attention prebuilt binary not found. PyTorch fallback active."

# Install vLLM for GRPO fast generation test
try:
    !pip install -q vllm
    print("vLLM installed.")
except Exception as exc:
    print(f"vLLM install skipped: {exc}")


In [ ]:
# Step 4: Load Qwen 3.5 2B Instruct with LoRA in non-thinking mode
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

# Note: In Qwen 3.5, 'Qwen/Qwen3.5-2B' is the official instruct/chat model on Hugging Face.
# 'Qwen/Qwen3.5-2B-Base' is the non-instruct base. We use non-thinking mode via enable_thinking=False.
MODEL_ID = "Qwen/Qwen3.5-2B"
print(f"Loading tokenizer for {MODEL_ID}...")

# Attempt loading with Unsloth if installed, else fallback to Transformers + PEFT
loaded_with_unsloth = False
try:
    from unsloth import FastLanguageModel
    print("Unsloth detected. Loading model with Unsloth fast kernels...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=512,
        dtype=torch.bfloat16,
        load_in_4bit=False,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    loaded_with_unsloth = True
    print("Model loaded with Unsloth successfully.")
except Exception as unsloth_err:
    print(f"Unsloth load skipped ({unsloth_err}). Loading with standard Transformers + PEFT...")

if not loaded_with_unsloth:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Test non-thinking mode formatting
sample_msg = [
    {"role": "system", "content": "You are Second, a chess assistant."},
    {"role": "user", "content": "Explain 1. e4."}
]

try:
    formatted_sample = tokenizer.apply_chat_template(
        sample_msg,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    print("Configured non-thinking mode using enable_thinking=False.")
except TypeError:
    formatted_sample = tokenizer.apply_chat_template(
        sample_msg,
        tokenize=False,
        add_generation_prompt=True
    )
    print("Formatted chat template without enable_thinking argument.")

print("Sample prompt template:")
print(formatted_sample[:160] + "...")

# Load base model with standard Transformers + PEFT if Unsloth is not used
if not loaded_with_unsloth:
    print(f"Loading base model {MODEL_ID} in bfloat16...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # Configure LoRA (rank 16, alpha 16, targeting all linear projections)
    peft_config = LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    print("Model ready with Unsloth fast PEFT configuration.")


In [ ]:
# Step 5: Prepare dummy chess data following the Second prompt contract
from datasets import Dataset

SYSTEM_PROMPT = "You are Second, a chess assistant. You explain chess positions and moves clearly using the provided facts."

DUMMY_SAMPLES = [
    {
        "prompt": """task: explain_move\nlevel: club\ndetail: short\n\nFEN: r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4\nWhite to move, move 4.\nMove played: Ng5.\nVerdict: Good.\nWhite's winning chances: 54% → 53%.\nBest move: d3.\nBest line: d3 Bc5 c3 d6.\nFacts:\n- Ng5 attacks the pawn on f7 together with the bishop on c4.\n- Black's best reply is d5.\n- Material: equal.\n- Phase: opening.\n- Opening: Italian Game, Two Knights Defense.""",
        "completion": "White plays Ng5 to put immediate pressure on the weak f7 pawn. The knight works with the bishop on c4 to create threats. Black must answer with d5 to close the diagonal and stay safe."
    },
    {
        "prompt": """task: explain_move\nlevel: beginner\ndetail: short\n\nFEN: r1bqkbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 2\nWhite to move, move 2.\nMove played: Nf3.\nVerdict: Best.\nWhite's winning chances: 52% → 52%.\nBest move: Nf3.\nBest line: Nf3 Nc6 Bb5 a6.\nFacts:\n- Nf3 develops the kingside knight toward the center.\n- Nf3 attacks the undefended pawn on e5.\n- Material: equal.\n- Phase: opening.\n- Opening: King's Pawn Game.""",
        "completion": "White develops the knight toward the center. This move directly attacks Black's undefended pawn on e5 and prepares kingside castling."
    },
    {
        "prompt": """task: explain_move\nlevel: club\ndetail: short\n\nFEN: r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/2P2N2/PP1P1PPP/RNBQ1RK1 b kq - 0 5\nBlack to move, move 5.\nMove played: Nxe4.\nVerdict: Mistake.\nBlack's winning chances: 48% → 32%.\nBest move: O-O.\nBest line: O-O d4 exd4 cxd4 Bb6.\nFacts:\n- Nxe4 captures the central pawn on e4.\n- White's best reply is d4, hitting the bishop and opening the center.\n- White has a lead in king safety.\n- Material: Black is up a pawn.\n- Phase: opening.\n- Opening: Italian Game, Giuoco Piano.""",
        "completion": "Black captures on e4 before castling. White strikes back with d4, opening lines toward Black's exposed king and regaining the initiative."
    },
    {
        "prompt": """task: explain_position\nlevel: club\ndetail: short\n\nFEN: 2r2rk1/1pq1bppp/p2p1n2/4p3/P3P3/1PN1BP2/1P4PP/2RQ1RK1 w - - 3 16\nWhite to move, move 16.\nBest move: Nd5.\nBest line: Nd5 Qd8 Nxe7+ Qxe7 Qd2.\nFacts:\n- White has an outpost on d5.\n- Nd5 attacks the black queen on c7.\n- Black's d6 pawn is backward on an open file.\n- Material: equal.\n- Phase: middlegame.""",
        "completion": "White controls the strong central outpost on d5. Playing Nd5 attacks the queen on c7 and increases pressure against Black's backward d6 pawn."
    }
]

# Build training dataset (30 rows)
dataset_rows = []
for i in range(30):
    item = DUMMY_SAMPLES[i % len(DUMMY_SAMPLES)]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["completion"]}
    ]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
    except TypeError:
        text = tokenizer.apply_chat_template(messages, tokenize=False)
    dataset_rows.append({"text": text, "prompt": item["prompt"], "completion": item["completion"]})

sft_dataset = Dataset.from_list(dataset_rows)
print(f"Prepared SFT dataset with {len(sft_dataset)} rows.")


In [ ]:
# Step 6: Run 10 steps of Supervised Fine-Tuning (SFT)
import time
from trl import SFTTrainer, SFTConfig
from transformers import Trainer

print("Starting 10-step SFT benchmark...")
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    max_steps=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_length=512,
    dataset_text_field="text",
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    args=sft_config,
)

# Use standard Transformers Trainer loss to bypass Triton fused_lm_head bug
if hasattr(model, "model_forward"):
    model.forward = model.model_forward

sft_trainer.compute_loss = lambda model, inputs, return_outputs=False, num_items_in_batch=None: (
    Trainer.compute_loss(sft_trainer, model, inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch)
)

start_sft = time.time()
sft_trainer.train()
sft_total_time = time.time() - start_sft

sft_peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)
sft_time_per_step = sft_total_time / 10.0

print("
--- SFT (10 Steps) Results ---")
print(f"Total Time:        {sft_total_time:.2f} s")
print(f"Time per Step:     {sft_time_per_step:.2f} s")
print(f"Peak GPU Memory:   {sft_peak_vram_gb:.2f} GB")



In [ ]:
# Step 7: Run 10 steps of GRPO with vLLM generation test and fallback
!pip install -q triton
import importlib
import transformers.utils as tu_trans
tu_trans._triton_available = True
import trl.kernels.chunked_logprob as clp
import trl.trainer.utils as tu
try:
    importlib.reload(clp)
    if hasattr(clp, "ChunkedLogProbFunction"):
        tu._ChunkedLogProbFunction = clp.ChunkedLogProbFunction
except Exception:
    pass

from trl import GRPOTrainer, GRPOConfig

print("Configuring GRPO test...")

# Dummy rule reward: reward = 1.0 if output is under 50 words, else 0.0
def dummy_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            text = completion[0]["content"] if len(completion) > 0 and isinstance(completion[0], dict) else str(completion)
        else:
            text = str(completion)
        word_count = len(text.strip().split())
        rewards.append(1.0 if word_count < 50 else 0.0)
    return rewards

# Build RL prompt dataset
rl_rows = []
for i in range(25):
    item = DUMMY_SAMPLES[i % len(DUMMY_SAMPLES)]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["prompt"]}
    ]
    try:
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    rl_rows.append({"prompt": formatted_prompt})

rl_dataset = Dataset.from_list(rl_rows)

vllm_active = False
grpo_total_time = 0.0
grpo_time_per_step = 0.0
grpo_peak_vram_gb = 0.0

# Test 1: Attempt fast generation with vLLM
print("\n[1/2] Attempting GRPO with vLLM fast generation (use_vllm=True)...")
try:
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    grpo_config_vllm = GRPOConfig(
        output_dir="./grpo_test_output",
        max_steps=10,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        num_generations=4,
        max_completion_length=128,
        learning_rate=5e-6,
        logging_steps=1,
        save_strategy="no",
        bf16=torch.cuda.is_bf16_supported(),
        use_vllm=True,
        vllm_gpu_memory_utilization=0.3,
        report_to="none"
    )
    
    trainer_vllm = GRPOTrainer(
        model=model,
        reward_funcs=[dummy_reward_func],
        args=grpo_config_vllm,
        train_dataset=rl_dataset,
    )
    
    start_grpo = time.time()
    trainer_vllm.train()
    grpo_total_time = time.time() - start_grpo
    grpo_peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)
    grpo_time_per_step = grpo_total_time / 10.0
    vllm_active = True
    print("vLLM fast generation succeeded.")
except Exception as exc:
    print(f"vLLM fast generation failed: {exc}")
    print("Proceeding to fallback generation test...")
    vllm_active = False

# Test 2: Standard TRL generation fallback if vLLM failed
if not vllm_active:
    print("\n[2/2] Running GRPO with standard TRL generation (use_vllm=False)...")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    grpo_config_standard = GRPOConfig(
        output_dir="./grpo_test_output",
        max_steps=10,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        num_generations=4,
        max_completion_length=128,
        learning_rate=5e-6,
        logging_steps=1,
        save_strategy="no",
        bf16=torch.cuda.is_bf16_supported(),
        use_vllm=False,
        report_to="none"
    )
    
    trainer_standard = GRPOTrainer(
        model=model,
        reward_funcs=[dummy_reward_func],
        args=grpo_config_standard,
        train_dataset=rl_dataset,
    )
    
    start_grpo = time.time()
    trainer_standard.train()
    grpo_total_time = time.time() - start_grpo
    grpo_peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)
    grpo_time_per_step = grpo_total_time / 10.0

print("\n--- GRPO (10 Steps) Results ---")
print(f"vLLM Generation:   {'Supported and Active' if vllm_active else 'Fallback to Standard TRL'}")
print(f"Total Time:        {grpo_total_time:.2f} s")
print(f"Time per Step:     {grpo_time_per_step:.2f} s")
print(f"Peak GPU Memory:   {grpo_peak_vram_gb:.2f} GB")



In [ ]:
# Step 8: Save checkpoint and full trainer state to Google Drive
print(f"Saving full trainer state and adapter to: {CHECKPOINT_DIR}")
sft_trainer.save_model(os.path.join(CHECKPOINT_DIR, "final_adapter"))
sft_trainer.save_state()
tokenizer.save_pretrained(CHECKPOINT_DIR)

print("Files saved to Drive:")
for fname in sorted(os.listdir(CHECKPOINT_DIR)):
    print(f" - {fname}")


In [ ]:
# Step 9: Verify Resumption from Checkpoint
# Handles both fresh runtime after restart and continuing current session
import os
import glob
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

CHECKPOINT_DIR = "/content/drive/MyDrive/second/checkpoints/phase0_test"
MODEL_ID = "Qwen/Qwen3.5-2B"

print(f"Checking for saved checkpoints in: {CHECKPOINT_DIR}")
checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*"))
latest_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1] if checkpoints else None
print(f"Latest checkpoint found: {latest_checkpoint}")

# Ensure tokenizer, dataset, and model exist if runtime was restarted
if "tokenizer" not in locals() or tokenizer is None:
    print("Initializing tokenizer after runtime restart...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

if "sft_dataset" not in locals() or sft_dataset is None:
    print("Rebuilding training sample after runtime restart...")
    dummy_sample = {
        "text": [
            """<|im_start|>system
You are Second, a chess assistant.<|im_end|>
<|im_start|>user
Explain 1. e4.<|im_end|>
<|im_start|>assistant
1. e4 attacks the center squares d5 and f5 and opens lines for White's bishop and queen.<|im_end|>"""
        ] * 16
    }
    sft_dataset = Dataset.from_dict(dummy_sample)

if "model" not in locals() or model is None:
    print("Reloading base model after runtime restart...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    peft_config = LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(base_model, peft_config)

if latest_checkpoint:
    print("Resuming training from checkpoint with restored optimizer and scheduler state...")
    resume_config = SFTConfig(
        output_dir=CHECKPOINT_DIR,
        max_steps=12,
        per_device_train_batch_size=2,
        learning_rate=1e-4,
        max_length=512,
        dataset_text_field="text",
        report_to="none"
    )
    resume_trainer = SFTTrainer(
        model=model,
        train_dataset=sft_dataset,
        args=resume_config,
    )

    from transformers import Trainer
    if hasattr(model, "model_forward"):
        model.forward = model.model_forward
    resume_trainer.compute_loss = lambda model, inputs, return_outputs=False, num_items_in_batch=None: (
        Trainer.compute_loss(resume_trainer, model, inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch)
    )

    t_resume_start = time.time()
    resume_trainer.train(resume_from_checkpoint=latest_checkpoint)
    resume_duration = time.time() - t_resume_start
    resumption_verified = True
    print(f"Resumption with optimizer state verified in {resume_duration:.2f} s.")
else:
    print("Adapter reload test as fallback...")
    from peft import PeftModel
    fresh_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    resumed_model = PeftModel.from_pretrained(fresh_base, os.path.join(CHECKPOINT_DIR, "final_adapter"), is_trainable=True)
    resumption_verified = True
    print("Adapter reload verified.")



In [ ]:
# Step 10: Print Markdown summary for docs/decisions/compute.md
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unknown GPU"
l4_rate_per_hr = 2.5 # Colab compute units per hour for L4

# Projections for Phase 5 (1,000 steps) and Phase 6 (500 steps)
sft_steps_target = 1000
grpo_steps_target = 500

sft_hours = (sft_time_per_step * sft_steps_target) / 3600.0
grpo_hours = (grpo_time_per_step * grpo_steps_target) / 3600.0

sft_units = sft_hours * l4_rate_per_hr
grpo_units = grpo_hours * l4_rate_per_hr

summary_table = f"""
## Phase 0.3 Training Stack Benchmark Results

| Metric | Recorded Value |
|---|---|
| **GPU Hardware** | {gpu_name} (24 GB VRAM) |
| **Base Model** | {MODEL_ID} |
| **LoRA Config** | Rank 16, Alpha 16 (all linear layers) |
| **SFT Time (10 steps)** | {sft_total_time:.2f} s ({sft_time_per_step:.2f} s/step) |
| **SFT Peak VRAM** | {sft_peak_vram_gb:.2f} GB |
| **GRPO Time (10 steps)** | {grpo_total_time:.2f} s ({grpo_time_per_step:.2f} s/step) |
| **GRPO Peak VRAM** | {grpo_peak_vram_gb:.2f} GB |
| **vLLM Fast Generation** | {'Supported and active' if vllm_active else 'Unsupported / Fallback to standard TRL'} |
| **Checkpoint Resumption** | {'Verified (2 steps completed)' if resumption_verified else 'Failed'} |

### Projected Compute Budget (Colab L4 at {l4_rate_per_hr} units/hr)

| Stage | Steps | Estimated Time | Estimated Colab Units |
|---|---|---|---|
| **Phase 5: SFT (2B)** | {sft_steps_target} | {sft_hours:.2f} hours | {sft_units:.1f} units |
| **Phase 6: GRPO (2B)** | {grpo_steps_target} | {grpo_hours:.2f} hours | {grpo_units:.1f} units |
| **Total 2B Training** | | **{sft_hours + grpo_hours:.2f} hours** | **{sft_units + grpo_units:.1f} units** |
"""

print(summary_table)
